# Model Training: Ford Used Car Price Prediction

This notebook trains and compares classical regression models for predicting Ford used car prices. The focus is on baseline model comparison, Random Forest tuning, model evaluation, and saving the best model.

In [ ]:
from pathlib import Path
import sys

import joblib
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from evaluate_models import compare_models, evaluate_regression_model
from model_tuning import tune_random_forest
from train_models import train_all_models
from visualization import (
    plot_actual_vs_predicted,
    plot_feature_importance,
    plot_residual_distribution,
)

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "ford_cleaned.csv"
METRICS_PATH = PROJECT_ROOT / "results" / "metrics" / "model_comparison.csv"
PLOTS_DIR = PROJECT_ROOT / "results" / "plots"
MODEL_PATH = PROJECT_ROOT / "models" / "best_model.pkl"

METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

## Load Processed Dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
print("Dataset shape:", df.shape)
print(df.dtypes)

## Define Features and Target

In [ ]:
X = df.drop(columns=["price"])
y = df["price"]

categorical_columns = X.select_dtypes(include=["object", "category"]).columns.tolist()

if categorical_columns:
    X = pd.get_dummies(X, columns=categorical_columns, drop_first=True)

feature_names = X.columns.tolist()

print("Number of features:", X.shape[1])
print("Categorical columns encoded:", categorical_columns)

The processed dataset still contains categorical columns such as model, transmission, and fuel type. These are converted with one-hot encoding before training. If the dataset is already fully numeric, this step leaves it unchanged.

## Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

## Train Baseline Models

In [ ]:
models = train_all_models(X_train, y_train)
models.keys()

The baseline comparison includes Linear Regression, Decision Tree, Random Forest, and Gradient Boosting. These are classical regression models suitable for a university machine learning project.

## Evaluate Baseline Models

In [ ]:
comparison_df = compare_models(models, X_test, y_test)
comparison_df = comparison_df.sort_values(["R2", "RMSE"], ascending=[False, True]).reset_index(drop=True)
comparison_df

Models are compared using MAE, RMSE, and R2 Score. Lower MAE/RMSE means smaller prediction errors, while higher R2 means the model explains more variance in car prices.

## Tune Random Forest

In [ ]:
tuned_random_forest, best_params, best_cv_score = tune_random_forest(X_train, y_train)

print("Best Random Forest parameters:", best_params)
print("Best cross-validation R2:", best_cv_score)

In [ ]:
models["Tuned Random Forest"] = tuned_random_forest

comparison_df = compare_models(models, X_test, y_test)
comparison_df = comparison_df.sort_values(["R2", "RMSE"], ascending=[False, True]).reset_index(drop=True)
comparison_df.to_csv(METRICS_PATH, index=False)
comparison_df

The tuned Random Forest is added to the comparison table after GridSearchCV. The table is saved to `results/metrics/model_comparison.csv` so the experiment results can be reviewed later.

## Select and Save Best Model

In [ ]:
best_model_name = comparison_df.iloc[0]["Model"]
best_model = models[best_model_name]
best_predictions = best_model.predict(X_test)

joblib.dump(best_model, MODEL_PATH)

print("Best model:", best_model_name)
print("Saved model to:", MODEL_PATH)

The best model is selected by highest R2 Score and then lowest RMSE. Linear Regression may underperform because car prices have non-linear relationships with mileage, age, model, engine size, and categorical features. Ensemble models such as Random Forest and Gradient Boosting often perform better because they can capture non-linear patterns and interactions between features.

## Best Model Evaluation

In [ ]:
best_metrics = evaluate_regression_model(best_model, X_test, y_test)
best_metrics

## Actual vs Predicted Plot

In [ ]:
actual_vs_predicted_path = plot_actual_vs_predicted(
    y_test,
    best_predictions,
    PLOTS_DIR / "best_model_actual_vs_predicted.png",
)
actual_vs_predicted_path

This plot shows how close the predicted prices are to the real prices. Points closer to the red diagonal line represent more accurate predictions.

## Residual Distribution Plot

In [ ]:
residual_plot_path = plot_residual_distribution(
    y_test,
    best_predictions,
    PLOTS_DIR / "best_model_residual_distribution.png",
)
residual_plot_path

Residuals show prediction errors. A residual distribution centered near zero suggests the model is not consistently overpredicting or underpredicting prices.

## Feature Importance Plot

In [ ]:
feature_importance_path = plot_feature_importance(
    best_model,
    feature_names,
    PLOTS_DIR / "best_model_feature_importance.png",
    top_n=15,
)
feature_importance_path

Feature importance is available for tree-based models and coefficient-based models. It helps identify which variables had the strongest influence on the best model's predictions.